# 🧠 NeuroAds — Brain Response Predictor
### Powered by Meta FAIR TRIBE v2 | Google Colab Runner

**Instructions:**
1. Run every cell top-to-bottom (Runtime → Run all).
2. After Cell 5 completes, click the **ngrok public URL** printed in the output.

> ⚡ Using a **GPU runtime** (Runtime → Change runtime type → T4 GPU) makes predictions ~10× faster.

## Cell 1 — Install system dependencies (FFmpeg)

In [ ]:
import subprocess, sys, os

print('Installing system libraries (FFmpeg, libGL)...')
subprocess.run(['apt-get', 'update', '-y'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-q', 'ffmpeg', 'libsm6', 'libxext6'], check=True)
print('✅ System dependencies installed')

## Cell 2 — Clone the NeuroAds repo from GitHub

In [ ]:
import os, subprocess, sys, shutil

REPO_URL = 'https://github.com/Muralidevml/Tribe-V2-Visualise-the-Neural-Pulse.git'
REPO_DIR = '/content/neuroads'

# ── Reset Working Directory ──
try:
    current_dir = os.getcwd()
except:
    os.chdir('/content')
    current_dir = '/content'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print(f'✅ Working directory: {os.getcwd()}')

## Cell 3 — Install Python dependencies

In [ ]:
print('Installing all dependencies (this may take 2-3 minutes)...')

# ── We install everything in ONE command so pip can resolve conflicts ──────
try:
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', 
        '-r', 'requirements.txt', 
        'pyngrok', 
        'tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git'
    ], check=True)
    print('\n✅ All packages installed successfully!')
except subprocess.CalledProcessError as e:
    print(f'\n❌ INSTALLATION FAILED: {e}')
    print('Try Runtime -> Factory Reset Runtime and run again.')

## Cell 4 — (Optional) HuggingFace login for model download

In [ ]:
from huggingface_hub import login
import os

HF_TOKEN = ''  # ← paste your token here if needed

if HF_TOKEN:
    login(token=HF_TOKEN)
    print('✅ Logged in to HuggingFace')
else:
    print('⚠️ No HF_TOKEN provided. If predictions fail, set your token here.')

## Cell 5 — Launch NeuroAds with public ngrok URL

In [ ]:
import subprocess, time, os, requests, sys
from pyngrok import ngrok

os.chdir(REPO_DIR)

print('🧹 Cleaning up old sessions...')
ngrok.kill()

for folder in ['uploads', 'cache', 'results', 'model']:
    os.makedirs(folder, exist_ok=True)

print('⏳ Starting Flask server (loading model weights)...')
with open('flask_log.txt', 'w') as log:
    subprocess.Popen(
        [sys.executable, 'app.py'],
        stdout=log, stderr=log, 
        start_new_session=True
    )

# Wait for Flask
retries = 30
ready = False
while retries > 0:
    try:
        requests.get('http://127.0.0.1:5000', timeout=2)
        ready = True
        break
    except:
        time.sleep(2)
        retries -= 1
        if retries % 5 == 0: print(f'   ...still loading ({retries*2}s elapsed)')

if not ready:
    print('❌ Flask failed to start. Check flask_log.txt')
else:
    public_url = ngrok.connect(5000)
    print(f'\n✅ NeuroAds is live!')
    print(f'👉 Open this URL: {public_url}')